In [1]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt


def save_nc_images(
    nc_folder,
    var="ssha",
    dpi=300
):
    """
    Convert all NetCDF files in nc_folder to PNG images.

    Creates a sister directory automatically:
        swaths/        -> swaths_images/
        data/swaths/   -> data/swaths_images/

    Parameters
    ----------
    nc_folder : str
        Folder containing .nc files.
    var : str
        Variable name to visualize.
    dpi : int
        DPI for saved PNGs.
    """

    # ------------------------------------------------------------
    # CREATE SISTER DIRECTORY
    # ------------------------------------------------------------
    parent_dir = os.path.dirname(os.path.abspath(nc_folder))
    folder_name = os.path.basename(os.path.normpath(nc_folder))

    image_dir = os.path.join(parent_dir, f"{folder_name}_images")
    os.makedirs(image_dir, exist_ok=True)

    # ------------------------------------------------------------
    # IMAGE SAVER
    # ------------------------------------------------------------
    def save_image(ssha, out_path):
        ssha = np.array(ssha, dtype=float)
    
        ssha[ssha > 9] = np.nan
    
        vmin = np.nanmin(ssha)
        vmax = np.nanmax(ssha)
    
        img = (ssha - vmin) / (vmax - vmin + 1e-8)
        img = np.clip(img, 0, 1)
    
        img = np.rot90(img, 2)
    
        plt.figure(figsize=(4, 10))
        plt.imshow(
            img,
            cmap="turbo",
            origin="lower",
            aspect="equal"
        )
    
        plt.axis("off")
        plt.tight_layout(pad=0)
        plt.savefig(out_path, dpi=dpi, bbox_inches="tight", pad_inches=0)
        plt.close()

    # ------------------------------------------------------------
    # PROCESS ALL NC FILES
    # ------------------------------------------------------------
    nc_files = sorted(
        f for f in os.listdir(nc_folder)
        if f.endswith(".nc")
    )

    print(f"Found {len(nc_files)} NetCDF files")

    for i, fname in enumerate(nc_files, start=1):

        nc_path = os.path.join(nc_folder, fname)

        try:
            ds = xr.open_dataset(nc_path)

            ssha = ds[var].values

            out_img = os.path.join(
                image_dir,
                os.path.splitext(fname)[0] + ".png"
            )

            save_image(ssha, out_img)

            ds.close()

            print(f"[{i}/{len(nc_files)}] Saved {out_img}")

        except Exception as e:
            print(f"Failed: {fname}")
            print(e)

    print("\nDone.")
    print(f"Images saved in:\n{image_dir}")

In [2]:
import os
from PIL import Image, ImageDraw


def draw_yolo_boxes(image_path, label_path, out_path):
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    img_w, img_h = img.size

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()

            if len(parts) < 5:
                continue

            class_id = parts[0]
            x_center, y_center, w, h = map(float, parts[1:5])

            # Convert YOLO normalized coords to pixel coords
            x_center *= img_w
            y_center *= img_h
            w *= img_w
            h *= img_h

            x1 = x_center - w / 2
            y1 = y_center - h / 2
            x2 = x_center + w / 2
            y2 = y_center + h / 2

            draw.rectangle(
                [x1, y1, x2, y2],
                outline="red",
                width=3
            )

            draw.text(
                (x1, y1),
                str(class_id),
                fill="red"
            )

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    img.save(out_path)

In [3]:
dataset_dir = "SWOT_IW_Labeled_Dataset"

for basin in os.listdir(dataset_dir):

    basin_path = os.path.join(dataset_dir, basin)

    if not os.path.isdir(basin_path):
        continue

    data_dir = os.path.join(basin_path, "data")
    label_dir = os.path.join(basin_path, "bb_labels")
    image_dir = os.path.join(basin_path, "data_images")
    out_dir = os.path.join(basin_path, "data_images_with_bb")

    if not os.path.isdir(data_dir):
        continue

    if not os.path.isdir(label_dir):
        continue

    print(f"Creating images from {data_dir}")
    save_nc_images(data_dir)

    os.makedirs(out_dir, exist_ok=True)

    for image_name in os.listdir(image_dir):

        if not image_name.endswith(".png"):
            continue

        base_name = os.path.splitext(image_name)[0]

        image_path = os.path.join(image_dir, image_name)
        label_path = os.path.join(label_dir, base_name + ".txt")
        out_path = os.path.join(out_dir, image_name)

        if not os.path.isfile(label_path):
            print(f"No label found for {image_name}")
            continue

        print(f"Adding boxes to {image_name}")
        draw_yolo_boxes(image_path, label_path, out_path)

Creating images from SWOT_IW_Labeled_Dataset/Andaman Sea/data
Found 198 NetCDF files
[1/198] Saved /Users/aguilarj/projects/swot-yolo/SWOT_IW_Labeled_Dataset/Andaman Sea/data_images/SWOT_L2_LR_SSH_Expert_001_202_20230728T095437_20230728T104606_PGC0_01.png
[2/198] Saved /Users/aguilarj/projects/swot-yolo/SWOT_IW_Labeled_Dataset/Andaman Sea/data_images/SWOT_L2_LR_SSH_Expert_001_383_20230803T210721_20230803T215801_PGC0_01.png
[3/198] Saved /Users/aguilarj/projects/swot-yolo/SWOT_IW_Labeled_Dataset/Andaman Sea/data_images/SWOT_L2_LR_SSH_Expert_001_411_20230804T210804_20230804T215833_PGC0_01.png
[4/198] Saved /Users/aguilarj/projects/swot-yolo/SWOT_IW_Labeled_Dataset/Andaman Sea/data_images/SWOT_L2_LR_SSH_Expert_001_439_20230805T210826_20230805T215904_PGC0_01.png
[5/198] Saved /Users/aguilarj/projects/swot-yolo/SWOT_IW_Labeled_Dataset/Andaman Sea/data_images/SWOT_L2_LR_SSH_Expert_001_508_20230808T081726_20230808T090812_PGC0_01.png
[6/198] Saved /Users/aguilarj/projects/swot-yolo/SWOT_IW_Lab